# Employee Order Processing Analysis

In [1]:
import pymysql
import pandas as pd

## Database Connection

Connect to the ClassicModels MySQL database.

In [2]:
conn = pymysql.connect(
    host='localhost',
    user='nhy',
    password='Hoangyen9626!',
    database='classicmodels'
)

print("Connected successfully")

Connected successfully


## Load Employee Data

Retrieve employee information from the employees table.

In [3]:
employees = pd.read_sql(
    """
    SELECT employeeNumber,
           firstName,
           lastName
    FROM employees
    """,
    conn
)

employees.head()

/var/folders/v6/s8fq5ntn2ms_7tyvk6l48qf00000gn/T/ipykernel_73389/185107561.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  employees = pd.read_sql(


,employeeNumber,firstName,lastName
0,1002,Diane,Murphy
1,1056,Mary,Patterson
2,1076,Jeff,Firrelli
3,1088,William,Patterson
4,1102,Gerard,Bondur


## Load Customer Data

Retrieve customer information and assigned sales representatives.

In [4]:
customers = pd.read_sql(
    """
    SELECT customerNumber,
           salesRepEmployeeNumber
    FROM customers
    """,
    conn
)

customers.head()

/var/folders/v6/s8fq5ntn2ms_7tyvk6l48qf00000gn/T/ipykernel_73389/1231012105.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customers = pd.read_sql(


,customerNumber,salesRepEmployeeNumber
0,125,NaN
1,169,NaN
2,206,NaN
3,223,NaN
4,237,NaN


## Load Order Data

Retrieve order information from the orders table.

In [5]:
orders = pd.read_sql(
    """
    SELECT orderNumber,
           customerNumber
    FROM orders
    """,
    conn
)

orders.head()

/var/folders/v6/s8fq5ntn2ms_7tyvk6l48qf00000gn/T/ipykernel_73389/2986711135.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  orders = pd.read_sql(


,orderNumber,customerNumber
0,10123,103
1,10298,103
2,10345,103
3,10124,112
4,10278,112


## Merge Employees and Customers

Match employees with the customers they manage.

In [6]:
emp_cust = pd.merge(
    employees,
    customers,
    left_on="employeeNumber",
    right_on="salesRepEmployeeNumber",
    how="inner"
)

emp_cust.head()

,employeeNumber,firstName,lastName,customerNumber,salesRepEmployeeNumber
0,1165,Leslie,Jennings,124,1165.0
1,1165,Leslie,Jennings,129,1165.0
2,1165,Leslie,Jennings,161,1165.0
3,1165,Leslie,Jennings,321,1165.0
4,1165,Leslie,Jennings,450,1165.0


## Merge with Orders

Connect employee-customer data with orders.

In [7]:
df = pd.merge(
    emp_cust,
    orders,
    on="customerNumber",
    how="inner"
)

df.head()

,employeeNumber,firstName,lastName,customerNumber,salesRepEmployeeNumber,orderNumber
0,1165,Leslie,Jennings,124,1165.0,10113
1,1165,Leslie,Jennings,124,1165.0,10135
2,1165,Leslie,Jennings,124,1165.0,10142
3,1165,Leslie,Jennings,124,1165.0,10182
4,1165,Leslie,Jennings,124,1165.0,10229


## Create Employee Full Name

In [8]:
df["employeeName"] = (
    df["firstName"] + " " + df["lastName"]
)

df.head()

,employeeNumber,firstName,lastName,customerNumber,salesRepEmployeeNumber,orderNumber,employeeName
0,1165,Leslie,Jennings,124,1165.0,10113,Leslie Jennings
1,1165,Leslie,Jennings,124,1165.0,10135,Leslie Jennings
2,1165,Leslie,Jennings,124,1165.0,10142,Leslie Jennings
3,1165,Leslie,Jennings,124,1165.0,10182,Leslie Jennings
4,1165,Leslie,Jennings,124,1165.0,10229,Leslie Jennings


## Calculate Total Orders Processed

Count the number of orders handled by each employee.

In [9]:
df = df.groupby(
    ["employeeNumber", "employeeName"],
    as_index=False
)["orderNumber"].count()

df.head()

,employeeNumber,employeeName,orderNumber
0,1165,Leslie Jennings,34
1,1166,Leslie Thompson,14
2,1188,Julie Firrelli,14
3,1216,Steve Patterson,18
4,1286,Foon Yue Tseng,17


## Rename Columns

In [10]:
df = df.rename(columns={
    "employeeNumber": "Employee Number",
    "orderNumber": "Total Orders Processed"
})

df.head()

,Employee Number,employeeName,Total Orders Processed
0,1165,Leslie Jennings,34
1,1166,Leslie Thompson,14
2,1188,Julie Firrelli,14
3,1216,Steve Patterson,18
4,1286,Foon Yue Tseng,17


## Select Relevant Columns

In [11]:
df = df[[
    "Employee Number",
    "employeeName",
    "Total Orders Processed"
]]

df.head()

,Employee Number,employeeName,Total Orders Processed
0,1165,Leslie Jennings,34
1,1166,Leslie Thompson,14
2,1188,Julie Firrelli,14
3,1216,Steve Patterson,18
4,1286,Foon Yue Tseng,17


## Identify Top Employee

In [12]:
df = df.sort_values(
    by="Total Orders Processed",
    ascending=False
).head(1)

df

,Employee Number,employeeName,Total Orders Processed
7,1370,Gerard Hernandez,43


## Additional Analysis

In [13]:
df.describe()

,Employee Number,Total Orders Processed
count,1.0,1.0
mean,1370.0,43.0
std,NaN,NaN
min,1370.0,43.0
25%,1370.0,43.0
50%,1370.0,43.0
75%,1370.0,43.0
max,1370.0,43.0


## Export Results

In [14]:
df.to_csv(
    "jupyter_highestorderprocessed.txt",
    sep="\t",
    index=False
)

print("Export completed")

Export completed


In [15]:
pd.read_csv(
    "jupyter_highestorderprocessed.txt",
    sep="\t"
)

,Employee Number,employeeName,Total Orders Processed
0,1370,Gerard Hernandez,43
